# Extract some kiwi calls from wild field recordings
---

### Activate conda environment if needed

```conda activate av310```

### Import packages and define environment variables

In [ ]:
import os, glob
import sys
import logging
from pathlib import Path
import subprocess


from avianz.src.core.batch_processor import BatchProcessor, BatchProcessorCallbacks


# Setup logging in case of errors
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)

# Setup path configurations
CONFIG_DIR = r"D:\avianz\avianz\Config"                             # Parent directory of avianz config file
# CONFIG_FILE = r"D:\avianz\avianz\Config\AviaNZconfig.txt"         # Config file contains needed parameters; can be modified to taste

AVIANZ_SOURCE_DIR = Path(r"D:\avianz\avianz")                       # Directory containing main script (AviaNZ.py)
# INPUT_AUDIO_DIR = Path(r"F:\arus\Yiluo\C1\C1_2023-10-22")           # Directory containing source audio (.wav)
AUDIO_DIR = r"F:\arus\Yiluo\C1\C1_2023-10-22"                       # Alternate audio source (fewer files for testing purposes)

FILTER_DIR = r"D:\avianz\avianz\Filters"                            # Directory containing species-specific wavelet filters (.txt, .h5, .json)

# Check what files are in the filter directory
files = os.listdir(FILTER_DIR)
# Find available species
species_list = sorted(list(set([os.path.splitext(f)[0] for f in files if '.' in f and not f.startswith('__')])))

print("Available species:")
for s in species_list:
    print(f"- {s}")

# List the calls of our species of interest: North Island Brown Biwi
kiwi_list = [ 
    'Kiwi (Nth Is Brown)',
    'Kiwi (Nth Is Brown)_chp'
    ]


def validate_environment():
    """Validates that folders and necessary files exist before running processes."""
    if not AVIANZ_SOURCE_DIR.exists():
        raise FileNotFoundError(f"AviaNZ engine directory not found at: {AVIANZ_SOURCE_DIR}")
    if not (AVIANZ_SOURCE_DIR / "AviaNZ.py").exists():
        raise FileNotFoundError(f"Could not locate 'AviaNZ.py' source wrapper within {AVIANZ_SOURCE_DIR}")
    if not INPUT_AUDIO_DIR.exists():
        raise FileNotFoundError(f"Target audio input path does not exist: {INPUT_AUDIO_DIR}")
        
    wav_files = list(INPUT_AUDIO_DIR.glob("*.wav"))
    if not wav_files:
        logging.warning(f"No target .wav files discovered inside {INPUT_AUDIO_DIR}")
    else:
        logging.info(f"Discovered {len(wav_files)} target audio source files for processing.")


# This throws alot of errors so handle them all at once. 
class MockCallbacks:
    def __getattr__(self, name):
        def dummy(*args, **kwargs):
            # Print which UI element the processor is looking for
            print(f"DEBUG: Processor requested UI callback: {name}")
            # Reasonable defaults for common UI interactions
            if "confirm" in name or "ask" in name: return True
            return None
        return dummy

class SilentCallbacks(BatchProcessorCallbacks):
    """Bypasses UI prompts for automated background processing."""
    def ask_resume_analysis(self, message): return True
    def confirm_analysis_launch(self, message): return True
    def update_progress(self, current, total, message): print(f"[{current}/{total}] {message}")



Available species:
- Bittern
- Bittern_chp
- Kakapo
- Kiwi (Great Spotted)
- Kiwi (Little Spotted)
- Kiwi (Little Spotted)_21-13-34
- Kiwi (Little Spotted)_chp
- Kiwi (Little Spotted)_syll_M
- Kiwi (Nth Is Brown)
- Kiwi (Nth Is Brown)_chp
- Kiwi (Tokoeka Fiordland)
- LongTailedCuckoo
- Morepork
- Morepork_04-23-42
- Morepork_chp
- NZ Bats
- NZ Bats_NP
- NZBats_NP


### Build the processor by specifying what we want to search for

In [2]:
# Initialize processor
try: 
    validate_environment()

    processor = BatchProcessor(
        configdir=CONFIG_DIR,       # Set parent didrectory for config file
        directory=AUDIO_DIR,        # Set source directory for audio files
        recognisers=kiwi_list,      # Pass the species list to recogniser
        # filtersDir=FILTER_DIR,
        callbacks=SilentCallbacks(),
    )

    # # Setup callback
    # processor.callbacks = MockCallbacks() # Bypasses the confirmation GUI

    print(f"DEBUG: Filter directory: {processor.filtersDir}")

    # Check filter dictionary
    processor.FilterDicts = processor.ConfigLoader.filters(processor.filtersDir)
    if processor.FilterDicts is None:
        print("DEBUG: FilterDicts is None. Check console for 'Could not load filter' messages.")
    else:
        print("Keys found:", list(processor.FilterDicts.keys()))


    # Run classifier
    processor.process_files()

    logging.info(f"Classification complete. Annotations .data saved to {INPUT_AUDIO_DIR}")

except Exception as e:
    logging.error(f"Execution failed: {e}", exc_info=True)
    sys.exit(1)

2026-06-21 16:42:46,457 [INFO] Discovered 20 target audio source files for processing.
Loading software settings from file D:\avianz\avianz\Config\AviaNZconfig.txt
Loading call filters from folder D:\avianz\avianz\Config\../Filters
Loaded filters: ['Bittern_chp', 'Bittern', 'Kakapo', 'Kiwi (Great Spotted)', 'Kiwi (Little Spotted)_chp', 'Kiwi (Little Spotted)_syll_M', 'Kiwi (Little Spotted)', 'Kiwi (Nth Is Brown)_chp', 'Kiwi (Nth Is Brown)', 'Kiwi (Tokoeka Fiordland)', 'LongTailedCuckoo', 'Morepork_chp', 'Morepork', 'NZ Bats_NP', 'NZ Bats']
DEBUG: Filter directory: D:\avianz\avianz\Config\../Filters
Loading call filters from folder D:\avianz\avianz\Config\../Filters
Loaded filters: ['Bittern_chp', 'Bittern', 'Kakapo', 'Kiwi (Great Spotted)', 'Kiwi (Little Spotted)_chp', 'Kiwi (Little Spotted)_syll_M', 'Kiwi (Little Spotted)', 'Kiwi (Nth Is Brown)_chp', 'Kiwi (Nth Is Brown)', 'Kiwi (Tokoeka Fiordland)', 'LongTailedCuckoo', 'Morepork_chp', 'Morepork', 'NZ Bats_NP', 'NZ Bats']
Keys found: 

Traceback (most recent call last):
  File "D:\avianz\avianz\src\core\config_loader.py", line 115, in getNNmodels
    model = model_loader.loadModel(nn_name, dirnn)
  File "D:\avianz\avianz\src\models\model_loader.py", line 50, in loadModel
    loaded = torch.load(pth_path, map_location='cpu', weights_only=False)
  File "d:\miniconda-windows\envs\av310\lib\site-packages\torch\serialization.py", line 1579, in load
    return _load(
  File "d:\miniconda-windows\envs\av310\lib\site-packages\torch\serialization.py", line 2190, in _load
    result = unpickler.load()
  File "d:\miniconda-windows\envs\av310\lib\site-packages\torch\serialization.py", line 2179, in find_class
    return super().find_class(mod_name, name)
ModuleNotFoundError: No module named 'src'
Traceback (most recent call last):
  File "D:\avianz\avianz\src\core\config_loader.py", line 115, in getNNmodels
    model = model_loader.loadModel(nn_name, dirnn)
  File "D:\avianz\avianz\src\models\model_loader.py", line 50, in loadMo

File loaded in 4.162410497665405
Working with recogniser: {'species': 'Kiwi (Nth Is Brown)', 'SampleRate': 16000, 'Filters': [{'calltype': 'Male', 'TimeRange': [3.0, 36.74, 0.76, 2.0], 'FreqRange': [900, 6000], 'WaveletParams': {'thr': 0.4, 'M': 0.76, 'nodes': [17, 43, 36, 44]}, 'ClusterCentre': [38.90659381726949, 10.354859942191352, 43.485525102177746, 0.0002527730734162275, 0.08036336995817471, 5.077241973894383, 1.562837796987371, 0.002956200748114056, 0.5248591739811833, 35.90933131853292, 3.024319759626494, 8.203591845481048, 2.1584204886253677, 39.475791372502215, 3.9605128181965727, 0.00021068927355561733, 0.00022970391533206887, 0.08059858333081071, 0.00029022641215918453, 4.3845522115377555, 0.699829072540868, 0.8357843424794329, 0.7387221928161879, 0.002683250604847691, 0.0017564828821649639, 0.5334690304495602, 0.003085083537104146, 25.033721588013847, 10.89493010529402, 1.2805559867175846, 1.7527983835700156, 4.500055508473179, 3.7115530356050597, 0.5589537066716067, 1.611